1. Principe de base : Estimation des erreurs via le $\chi^2$
On part d’un modèle dépendant de deux paramètres $\theta_1$ et $\theta_2$, et on cherche à estimer leurs incertitudes et leur covariance à partir de la minimisation du $\chi^2$.

Hypothèses

On a une fonction de coût $\chi^2(\theta_1, \theta_2)$  qui mesure l’écart entre les données et le modèle.
Au minimum du $\chi^2$, on peut approximer localement $\chi^2$ par une forme quadratique (valable si le minimum est bien défini et que les erreurs sont gaussiennes).

2. Développement limité du $\chi^2$ autour du minimum
Soit $\hat{\theta}_1, \hat{\theta}_2$ les valeurs qui minimisent $\chi^2$. On effectue un développement limité au 2nd ordre autour de ce minimum :

$$ \chi2(\theta_1,\theta_2) = \chi2_{min} + (\Delta \theta_1 \, \Delta \theta_2) H (\Delta \theta_1 \, \Delta \theta_2)$$

χ2(θ1​,θ2​)≈χmin2​+21​[Δθ1​​Δθ2​​][∂θ12​∂2χ2​∂θ2​∂θ1​∂2χ2​​∂θ1​∂θ2​∂2χ2​∂θ22​∂2χ2​​][Δθ1​Δθ2​​]

où $\Delta \theta_i = \theta_i - \hat{\theta_i}$

On note la **matrice hessienne** HHH :

$$H = \begin{bmatrix}  
\frac{\partial^2 \chi^2}{\partial \theta_1^2} & \frac{\partial^2 \chi^2}{\partial \theta_1 \partial \theta_2} \\  
\frac{\partial^2 \chi^2}{\partial \theta_2 \partial \theta_1} & \frac{\partial^2 \chi^2}{\partial \theta_2^2}  
\end{bmatrix}$$

## Method 1 : Define the chi2 function as a function of parameters

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import minimize

In [ ]:
# Exemple : fonction chi2 à minimiser (à adapter à ton problème)
def chi2(theta, data):
    theta1, theta2 = theta
    # Exemple : modèle linéaire y = theta1 * x + theta2
    x, y, y_err = data
    y_model = theta1 * x + theta2
    return np.sum(((y - y_model) / y_err) ** 2)

In [ ]:
# Données simulées
x = np.linspace(0, 10, 50)
y_true = 2 * x + 3
y_err = 0.5
y = y_true + y_err * np.random.randn(len(x))
data = (x, y, y_err)

In [ ]:
# Minimisation du chi2
result = minimize(chi2, x0=[1.0, 1.0], args=(data,))
theta_hat = result.x
print(f"Valeurs optimales : theta1 = {theta_hat[0]:.3f}, theta2 = {theta_hat[1]:.3f}")

In [ ]:
# Minimisation du chi2 avec une méthode qui calcule la hessienne
result = minimize(chi2, x0=[1.0, 1.0], args=(data,), method="trust-constr")
theta_hat = result.x
print(f"Valeurs optimales : theta1 = {theta_hat[0]:.3f}, theta2 = {theta_hat[1]:.3f}")

In [ ]:
# Récupération de la matrice hessienne
if hasattr(result, "hess"):
    hess = result.hess
else:
    # Calcul numérique de la hessienne
    from scipy.optimize import approx_fprime

    eps = 1e-5
    hess = np.zeros((2, 2))
    for i in range(2):
        for j in range(2):

            def chi2_i(theta):
                return chi2(theta, data)

            theta_plus = theta_hat.copy()
            theta_plus[i] += eps
            grad_plus = approx_fprime(theta_plus, chi2_i, eps)
            theta_minus = theta_hat.copy()
            theta_minus[i] -= eps
            grad_minus = approx_fprime(theta_minus, chi2_i, eps)
            hess[i, j] = (grad_plus[j] - grad_minus[j]) / (2 * eps)
    hess = 0.5 * hess  # Facteur 1/2 pour la matrice de covariance

In [ ]:
# Matrice de covariance
C = np.linalg.inv(hess)
print("Matrice de covariance :\n", C)

In [ ]:
# Après avoir calculé C (matrice de covariance)
sigma_theta1 = np.sqrt(C[0, 0])
sigma_theta2 = np.sqrt(C[1, 1])

In [ ]:
# Valeurs propres et vecteurs propres pour l'ellipse
eigvals, eigvecs = np.linalg.eig(C)
a = np.sqrt(eigvals[0])  # Demi-grand axe
b = np.sqrt(eigvals[1])  # Demi-petit axe
phi = np.arctan2(eigvecs[1, 0], eigvecs[0, 0])  # Angle de rotation

In [ ]:
# Tracé de l'ellipse
theta_ellipse = np.linspace(0, 2 * np.pi, 100)
ellipse_x = a * np.cos(theta_ellipse) * np.cos(phi) - b * np.sin(theta_ellipse) * np.sin(phi)
ellipse_y = a * np.cos(theta_ellipse) * np.sin(phi) + b * np.sin(theta_ellipse) * np.cos(phi)

plt.figure(figsize=(8, 6))
plt.scatter(x, y, label="Données", color="blue", alpha=0.5)
plt.plot(x, theta_hat[0] * x + theta_hat[1], label="Modèle ajusté", color="red")
plt.plot(theta_hat[0] + ellipse_x, theta_hat[1] + ellipse_y, label="Ellipse 1σ", color="green")
plt.xlabel("x")
plt.ylabel("y")
plt.legend()
plt.title("Ajustement et ellipse d'erreur à 1σ")
plt.grid(True)
plt.axis("equal")
plt.show()

## Method 2 : Start with function y = f(x)

In [ ]:
from scipy.optimize import curve_fit


# Modèle à ajuster
def model(x, theta1, theta2):
    return theta1 * x + theta2


# Ajustement
popt, pcov = curve_fit(model, x, y, sigma=y_err)
theta_hat = popt
C = pcov  # Matrice de covariance directement disponible

print(f"Valeurs optimales : theta1 = {theta_hat[0]:.3f}, theta2 = {theta_hat[1]:.3f}")
print("Matrice de covariance :\n", C)

In [ ]:
theta1_hat = popt[0]
theta2_hat = popt[1]

In [ ]:
# Erreurs standard
sigma_theta1 = np.sqrt(C[0, 0])
sigma_theta2 = np.sqrt(C[1, 1])

print(
    f"Valeurs ajustées : theta1 = {theta1_hat:.3f} ± {sigma_theta1:.4f}, theta2 = {theta2_hat:.3f} ± {sigma_theta2:.4f}"
)